# LLM Extraction Function Overview

This notebook was used to run the LLM extraction function on the article data.

The **main function**:
- **Inputs**: 
  - The dataset to process (CSV, or parquet file).
- **Functionality**:
  - Runs LLM extraction and returns a parsed DataFrame with:
    - Five types of entities:
      - `people`
      - `institutions`
      - `political_events`
      - `political_parties`
      - `locations`
    - Entity relationships:
      - `source`
      - `target`
      - `relationship`
      - `sentiment`
    - Relevance classification (`is_relevant`).
  - Parses the JSON output for each row into a structured DataFrame.

- **Prompt Construction**: Builds a dynamic prompt for each row, including the language and article text.

### Additional Features:
- **Language Filtering**: 
  - Processes only English, French, or German texts (or skips if not specified).
- **Backup Mechanism**:
  - Saves periodic backups to a folder.
  - On rerun, skips already processed rows.
- **Error Handling**:
  - Handles API rate limits and retries with a lag if limits are hit.
  - Logs errors for any failed requests.

### Output Example:
```json
{
  "entities": {
    "people": [],
    "institutions": [],
    "political_events": [],
    "political_parties": [],
    "locations": []
  },
  "entity_relationships": [
    {
      "source": "EntityA",
      "target": "EntityB",
      "relationship": "supports",
      "sentiment": "FRIENDLY"
    }
  ],
  "is_relevant": 1
}

In [8]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))

from lib.extract_with_llm import main

%load_ext autoreload
%autoreload 2

In [2]:
df_results = main('df_test.csv')

📥 Loading data...
📁 Created backup folder: backups


  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:51<00:00,  2.56s/it]

💾 Final results saved to backups/gemini_results_backup_20250519_0027_final.csv
✅ Finished all rows. Returning 20 results as DataFrame.


In [3]:
df_results.head(20)

,url,lang,extracted_entities,entity_relationships,relevant_to_german_or_french_elections
0,https://www.bfmtv.com/societe/au-chateau-de-ch...,fr,"{'people': [], 'institutions': ['château de Ch...",[],False
1,https://www.reuters.com/technology/litmus-test...,en,"{'people': ['Donald Trump'], 'institutions': [...","[{'source': 'Donald Trump', 'target': 'TSMC', ...",False
2,https://www.politico.eu/tag/irish-politics/,en,"{'people': ['Micheál Martin'], 'institutions':...","[{'source': 'Micheál Martin', 'target': 'Sinn ...",False
3,https://www.politico.eu/article/ukraine-plead-...,en,"{'people': ['Donald Trump', 'Zelenskyy', 'Lloy...","[{'source': 'Donald Trump', 'target': 'Ukraine...",False
4,https://www.lemonde.fr/en/economy/article/2025...,en,"{'people': ['Eric Lombard'], 'institutions': [...",[],False
5,https://www.france24.com/en/live-news/20250416...,en,"{'people': ['Aaron Boupendza'], 'institutions'...",[],False
6,https://www.reuters.com/business/energy/back-r...,en,"{'people': [], 'institutions': ['United States...",[],False
7,https://www.euronews.com/2025/04/08/global-exe...,en,"{'people': ['Agnès Callamard'], 'institutions'...","[{'source': 'Agnès Callamard', 'target': 'Iran...",False
8,https://www.euronews.com/2025/03/15/uk-to-hold...,en,"{'people': ['Keir Starmer', 'Vladimir Putin', ...","[{'source': 'Keir Starmer', 'target': 'Vladimi...",False
9,https://www.rfi.fr/en/international/20241214-f...,en,"{'people': [], 'institutions': ['Australian th...","[{'source': 'Australian think tank', 'target':...",False


In [4]:
df_results.iloc[19].entity_relationships

[{'source': 'SNCF Voyageurs',
  'target': 'Alstom',
  'relationship': 'accuses',
  'sentiment': 'HOSTILE'},
 {'source': 'Alstom',
  'target': 'SNCF Voyageurs',
  'relationship': 'partners with',
  'sentiment': 'NEUTRAL'},
 {'source': 'Île-de-France Mobilités (IDFM)',
  'target': 'RER D',
  'relationship': 'supports',
  'sentiment': 'NEUTRAL'},
 {'source': 'Île-de-France Mobilités (IDFM)',
  'target': 'RER E',
  'relationship': 'supports',
  'sentiment': 'NEUTRAL'}]